In [1]:
from dataclasses import dataclass
from enum import Enum, auto
import numpy as np

In [2]:
@dataclass
class Var:
    coefficient: float
    symbol: str

    def __repr__(self):
        return f"{self.coefficient}{self.symbol}"

In [3]:
class Op(Enum):
    LE = auto()
    LEQ = auto()
    GE = auto()
    GEQ = auto()
    EQ = auto()

    def __repr__(self):
        match self:
            case Op.LE:
                return "<"
            case Op.LEQ:
                return "<="
            case Op.GE:
                return ">"
            case Op.GEQ:
                return ">="
            case Op.EQ:
                return "="

In [4]:
# objective = 30x + 40y
objective = [Var(30, "x"), Var(40, "y")]

In [5]:
# constraints
# 1: 2x + y <= 10
c1 = ([Var(2, "x"), Var(1, "y")], Op.GEQ, 10)
# 2: x + y <= 7
c2 = ([Var(1, "x"), Var(1, "y")], Op.LEQ, 7)
# 3: x + 2y <= 12
c3 = ([Var(1, "x"), Var(2, "y")], Op.LEQ, 12)

constraints = [c1, c2, c3]

In [6]:
constraints

[([2x, 1y], >=, 10), ([1x, 1y], <=, 7), ([1x, 2y], <=, 12)]

In [7]:
# negate objective
for v in objective:
    v.coefficient *= -1
objective = (objective, Op.LEQ, 0)
objective

([-30x, -40y], <=, 0)

In [8]:
slacked_constraints = []

for i, constraint in enumerate(constraints):
    new_lhs = constraint[0].copy()
    for j in range(len(constraints)):
        coef = 1 if j == i else 0
        new_lhs.append(Var(coef, f"s{j+1}"))
    new_lhs.append(Var(0, f"p"))
    slacked_constraints.append((new_lhs, Op.EQ, constraint[2]))

slacked_constraints

new_lhs = objective[0].copy()
for j in range(len(constraints)):
    new_lhs.append(Var(0, f"s{j+1}"))
new_lhs.append(Var(1, f"p"))
slacked_objective = (new_lhs, Op.EQ, 0)

print("Objective")
print(slacked_objective)
print("Constraints")
print(slacked_constraints)

Objective
([-30x, -40y, 0s1, 0s2, 0s3, 1p], =, 0)
Constraints
[([2x, 1y, 1s1, 0s2, 0s3, 0p], =, 10), ([1x, 1y, 0s1, 1s2, 0s3, 0p], =, 7), ([1x, 2y, 0s1, 0s2, 1s3, 0p], =, 12)]


In [9]:
# Tableau Formation

In [11]:
main_rows = np.array([
    [v.coefficient for v in constraint[0]]
    for constraint in slacked_constraints
], dtype=float)
res_col = np.array([
    constraint[2]
    for constraint in slacked_constraints
], dtype=float)[:, None]
bottom_row = np.array([
    i.coefficient for i in slacked_objective[0]
], dtype=float)

In [12]:
tableau = np.vstack([
    np.hstack([main_rows, res_col]),
    np.append(bottom_row, 0)
])
tableau

array([[  2.,   1.,   1.,   0.,   0.,   0.,  10.],
       [  1.,   1.,   0.,   1.,   0.,   0.,   7.],
       [  1.,   2.,   0.,   0.,   1.,   0.,  12.],
       [-30., -40.,   0.,   0.,   0.,   1.,   0.]])

In [13]:
tableau[:-1, :-1]   # constraint coefficients

array([[2., 1., 1., 0., 0., 0.],
       [1., 1., 0., 1., 0., 0.],
       [1., 2., 0., 0., 1., 0.]])

In [14]:
tableau[:-1, -1, None]    # constraint RHS

array([[10.],
       [ 7.],
       [12.]])

In [15]:
tableau[-1, :-1]    # objective coefficients

array([-30., -40.,   0.,   0.,   0.,   1.])

In [16]:
tableau[-1, -1]     # objective/RHS corner

np.float64(0.0)

In [20]:
def pivot(input_tbl):
    tbl = input_tbl.copy()
    col_pivot = tbl[-1, :-1].argmin()
    print(tbl[-1, :-1])
    if tbl[-1, :-1][col_pivot] >= 0:
        print("No more pivots possible")
        return tbl, True
    print(f"Pivoting on index {col_pivot} ({tbl[-1, :-1][col_pivot]}) as its the lowest value")
    
    pivot_candidates = tbl[:-1, -1, None] / tbl[:-1, col_pivot, None]
    print(pivot_candidates)
    masked_pivots = np.where(pivot_candidates > 0, pivot_candidates, np.inf)
    row_pivot = masked_pivots.argmin()
    if np.all(np.isinf(masked_pivots)):
        raise Exception("Problem is unbounded")
    print(f"row pivot on {row_pivot}")
    
    print(f"Pivots: Row: {row_pivot}, Col: {col_pivot}")

    # Divide row by coefficient in pivot_col to reduce that coefficient to 1
    reduction_factor = tbl[row_pivot][col_pivot]
    tbl[row_pivot] = tbl[row_pivot] / reduction_factor

    print(tbl)
    # Now zero out coefficient in non pivot_row rows
    for idx, row in enumerate(tbl):
        if idx == row_pivot:
            continue
        coeff = row[col_pivot]
        row_to_add = (-1 * coeff) * tbl[row_pivot]
        tbl[idx] = row_to_add + row
    print(tbl)
    return tbl, False

In [21]:
tbl = tableau
while True:
    tbl, finished = pivot(tbl)
    if finished:
        final_tableau = tbl
        break

[-30. -40.   0.   0.   0.   1.]
Pivoting on index 1 (-40.0) as its the lowest value
[[10.]
 [ 7.]
 [ 6.]]
row pivot on 2
Pivots: Row: 2, Col: 1
[[  2.    1.    1.    0.    0.    0.   10. ]
 [  1.    1.    0.    1.    0.    0.    7. ]
 [  0.5   1.    0.    0.    0.5   0.    6. ]
 [-30.  -40.    0.    0.    0.    1.    0. ]]
[[  1.5   0.    1.    0.   -0.5   0.    4. ]
 [  0.5   0.    0.    1.   -0.5   0.    1. ]
 [  0.5   1.    0.    0.    0.5   0.    6. ]
 [-10.    0.    0.    0.   20.    1.  240. ]]
[-10.   0.   0.   0.  20.   1.]
Pivoting on index 0 (-10.0) as its the lowest value
[[ 2.66666667]
 [ 2.        ]
 [12.        ]]
row pivot on 1
Pivots: Row: 1, Col: 0
[[  1.5   0.    1.    0.   -0.5   0.    4. ]
 [  1.    0.    0.    2.   -1.    0.    2. ]
 [  0.5   1.    0.    0.    0.5   0.    6. ]
 [-10.    0.    0.    0.   20.    1.  240. ]]
[[  0.   0.   1.  -3.   1.   0.   1.]
 [  1.   0.   0.   2.  -1.   0.   2.]
 [  0.   1.   0.  -1.   1.   0.   5.]
 [  0.   0.   0.  20.  10.   1.

In [22]:
final_tableau

array([[  0.,   0.,   1.,  -3.,   1.,   0.,   1.],
       [  1.,   0.,   0.,   2.,  -1.,   0.,   2.],
       [  0.,   1.,   0.,  -1.,   1.,   0.,   5.],
       [  0.,   0.,   0.,  20.,  10.,   1., 260.]])

In [28]:
A = final_tableau[:, :-1]
A

array([[ 0.,  0.,  1., -3.,  1.,  0.],
       [ 1.,  0.,  0.,  2., -1.,  0.],
       [ 0.,  1.,  0., -1.,  1.,  0.],
       [ 0.,  0.,  0., 20., 10.,  1.]])

In [51]:
basic_cols = np.where(
    (A == 1).sum(axis=0) == 1
)[0]

In [58]:
basic_rows = np.argmax(A[:, basic_cols] == 1, axis=0)

In [63]:
basic_values = final_tableau[:, -1][basic_rows]

In [64]:
dict(zip(basic_cols, basic_values))

{np.int64(0): np.float64(2.0),
 np.int64(1): np.float64(5.0),
 np.int64(2): np.float64(1.0),
 np.int64(5): np.float64(260.0)}